In [ ]:
# @title Install dependencies
# Base deps suffice: precision/attention auto-selected per GPU.
# (flash-attn requires sm_80+ and deepspeed/unsloth are not used
#  by the library training path.)
!pip install -q -r requirements/base.txt
!pip install -q -e .
!pip install -q huggingface_hub[hf_transfer]
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# @title Login to Hugging Face
from huggingface_hub import login
import getpass
token = getpass.getpass("Enter HF_TOKEN: ")
login(token=token)

# Clone project
repo_id = "Kandil7/Baligh-1.7B"
!git clone https://huggingface.co/${repo_id} /content/Baligh 2>/dev/null || (
    git clone https://github.com/Kandil7/Baligh.git /content/Baligh
)

In [ ]:
# @title Prepare Data (run once)
cd /content/Baligh
!python -m src.scripts.prepare_data --stage cpt --clean --output-dir data/train_ready

In [ ]:
# @title Run CPT Training (auto-resumes from latest checkpoint)
!python -m src.scripts.run_cpt \
  --data-dir /content/Baligh/data/train_ready/cpt \
  --output-dir /content/Baligh/training/cpt \
  --eval-data /content/Baligh/data/train_ready/cpt_eval

# If Colab disconnects, re-run this cell — it auto-resumes!

# For full training (50K steps), use:
# !python -m src.scripts.run_cpt \
#   --data-dir /content/Baligh/data/train_ready/cpt \
#   --output-dir /content/Baligh/training/cpt \
#   --config configs/cpt/cpt-stage2.yaml

In [ ]:
# @title List Checkpoints
from baligh.training.checkpoint import CheckpointManager
manager = CheckpointManager("/content/Baligh/training/cpt")
checkpoints = manager.list_checkpoints()
if checkpoints:
    for cp in checkpoints:
        print(f"  Step {cp['step']:>6} | Loss: {cp.get('loss', 'N/A')} | {cp['path']}")
else:
    print("No checkpoints found")

In [ ]:
# @title Push CPT Model to HF Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="/content/Baligh/training/cpt/final",
    repo_id="Kandil7/Baligh-1.7B",
    repo_type="model",
    commit_message="CPT checkpoint",
    path_in_repo="cpt"
)